# Module 3 WWTP/RWZI overview and provisional spatial mapping

This notebook uses only the five `RWZI *.xlsx` workbooks plus the `RWZI` sheet in `Lozingspunten (1).xlsx`.

It does three things:
1. Combines `alle data` with TP, TN and COD for each WWTP.
2. Creates a site-level data overview.
3. Creates provisional straight-line distances from WWTP discharge points to candidate MPS measurement coordinates.

Important: these are **candidate spatial distances**, not final hydrological routing. Final routing still needs confirmed MP01-MP07 locations and river topology/residence time.


In [ ]:
from pathlib import Path

import pandas as pd
import geopandas as gpd


In [ ]:
DATA_DIR = Path('.')

WWTP_FILES = {
    'Achel': DATA_DIR / 'RWZI ACHEL.xlsx',
    'Eksel': DATA_DIR / 'RWZI eksel.xlsx',
    'Lommel': DATA_DIR / 'RWZI Lommel.xlsx',
    'Overpelt': DATA_DIR / 'RWZI overpelt.xlsx',
    'Peer': DATA_DIR / 'RWZI Peer.xlsx',
}

SHEETS = {
    'Achel': {'TP': 'TP', 'TN': 'TN', 'COD': 'COD'},
    'Eksel': {'TP': 'TP', 'TN': 'TN', 'COD': 'COD'},
    'Lommel': {'TP': 'tot P', 'TN': 'tot N', 'COD': 'tot CZV'},
    'Overpelt': {'TP': 'TP', 'TN': 'TN', 'COD': 'COD'},
    'Peer': {'TP': 'TP', 'TN': 'TN', 'COD': 'COD'},
}

LOZINGEN_FILE = DATA_DIR / 'Lozingspunten (1).xlsx'
MPS_GPKG_FILE = DATA_DIR / 'MPS_stations.gpkg'

OUTPUT_COMBINED_CSV = DATA_DIR / 'M5_Module3_WWTP_combined_15min.csv'
OUTPUT_DISTANCE_CSV = DATA_DIR / 'M5_Module3_WWTP_to_MP_candidate_distances.csv'
OUTPUT_EXCEL = DATA_DIR / 'M5_Module3_WWTP_overview_and_spatial_mapping.xlsx'

for path in list(WWTP_FILES.values()) + [LOZINGEN_FILE, MPS_GPKG_FILE]:
    if not path.exists():
        raise FileNotFoundError(f'File not found: {path}')


## 1. Functions to read one WWTP workbook

The separate `effluentASM`, `effluentTN`, and `effluentTP` files are not needed here, because the same data are already in the five RWZI workbooks.


In [ ]:
def clean_time_column(data, time_column='time'):
    """Convert a time column to datetime and drop invalid/unit rows."""
    result = data.copy()
    result[time_column] = pd.to_datetime(result[time_column], errors='coerce')
    result = result[result[time_column].notna()].copy()
    result[time_column] = result[time_column].dt.round('15min')
    result = result.drop_duplicates(subset=[time_column], keep='first')
    return result.reset_index(drop=True)


def read_concentration_sheet(excel_file, sheet_name, output_column):
    """Read a two-column TP, TN or COD sheet."""
    data = pd.read_excel(excel_file, sheet_name=sheet_name, usecols=[0, 1])

    data = data.rename(columns={
        data.columns[0]: 'time',
        data.columns[1]: output_column,
    })

    data = clean_time_column(data, 'time')
    data[output_column] = pd.to_numeric(data[output_column], errors='coerce')
    data = data[data[output_column].notna()].copy()

    return data[['time', output_column]].reset_index(drop=True)


def read_alle_data_sheet(excel_file, reference_times):
    """Read useful measurable variables from the 'alle data' sheet."""
    columns_to_read = [
        'time',
        'Flow rate',
        'Temperature',
        'Dissolved oxygen',
        'Ammonia nitrogen',
        'Nitrate nitrogen',
        'Orthophosphate',
        'Total suspended solids',
    ]

    data = pd.read_excel(excel_file, sheet_name='alle data', usecols=columns_to_read)

    data = data.rename(columns={
        'Flow rate': 'flow_m3d',
        'Temperature': 'temperature_degC',
        'Dissolved oxygen': 'do_mgL',
        'Ammonia nitrogen': 'nh4_mgN_L',
        'Nitrate nitrogen': 'no3_mgN_L',
        'Orthophosphate': 'opo4_mgP_L',
        'Total suspended solids': 'tss_mgL',
    })

    numeric_columns = [
        'flow_m3d', 'temperature_degC', 'do_mgL',
        'nh4_mgN_L', 'no3_mgN_L', 'opo4_mgP_L', 'tss_mgL',
    ]

    for column in numeric_columns:
        data[column] = pd.to_numeric(data[column], errors='coerce')

    # Remove unit rows and empty rows.
    data = data[data['flow_m3d'].notna()].copy().reset_index(drop=True)

    # Eksel has invalid year-0000 timestamps. Detect that before parsing dates.
    time_as_text = data['time'].astype(str)
    mostly_year_0000 = time_as_text.str.contains('0000', na=False).mean() > 0.90

    if mostly_year_0000:
        data['time'] = list(reference_times)[:len(data)]
        data['time_reconstructed_from_concentration_sheet'] = True
    else:
        data['time'] = pd.to_datetime(data['time'], errors='coerce')
        data['time_reconstructed_from_concentration_sheet'] = False
        data = data[data['time'].notna()].copy()

    data['time'] = pd.to_datetime(data['time']).dt.round('15min')
    data = data.drop_duplicates(subset=['time'], keep='first')

    return data.reset_index(drop=True)


In [ ]:
def combine_one_wwtp(site, file_path):
    """Combine alle data, TP, TN and COD for one WWTP."""
    with pd.ExcelFile(file_path) as excel_file:
        tp = read_concentration_sheet(excel_file, SHEETS[site]['TP'], 'tp_mgP_L')
        tn = read_concentration_sheet(excel_file, SHEETS[site]['TN'], 'tn_mgN_L')
        cod = read_concentration_sheet(excel_file, SHEETS[site]['COD'], 'cod_mgO2_L')
        alle_data = read_alle_data_sheet(excel_file, reference_times=tp['time'])

    combined = alle_data.merge(cod, on='time', how='outer')
    combined = combined.merge(tn, on='time', how='outer')
    combined = combined.merge(tp, on='time', how='outer')

    combined['wwtp_site'] = site
    combined['source_file'] = file_path.name

    return combined.sort_values('time').reset_index(drop=True)


def add_load_columns(data):
    """Add 15-minute volume and load columns."""
    result = data.copy()

    # Flow is m3/day. A 15-minute interval is 1/96 of one day.
    result['volume_m3_15min'] = result['flow_m3d'] / 96
    result['flow_m3s'] = result['flow_m3d'] / 86400

    result['cod_kg_15min'] = result['cod_mgO2_L'] * result['volume_m3_15min'] / 1000
    result['tn_kg_15min'] = result['tn_mgN_L'] * result['volume_m3_15min'] / 1000
    result['tp_kg_15min'] = result['tp_mgP_L'] * result['volume_m3_15min'] / 1000
    result['nh4_kg_15min'] = result['nh4_mgN_L'] * result['volume_m3_15min'] / 1000
    result['no3_kg_15min'] = result['no3_mgN_L'] * result['volume_m3_15min'] / 1000
    result['opo4_kg_15min'] = result['opo4_mgP_L'] * result['volume_m3_15min'] / 1000
    result['tss_kg_15min'] = result['tss_mgL'] * result['volume_m3_15min'] / 1000

    return result


## 2. Combine the five WWTP files

If your computer is slow with Excel files, run this cell site by site. The outputs should match the summary shown below.


In [ ]:
combined_tables = []

for site, file_path in WWTP_FILES.items():
    print(f'Reading {site}: {file_path.name}')
    site_data = combine_one_wwtp(site, file_path)
    combined_tables.append(site_data)

wwtp_combined = pd.concat(combined_tables, ignore_index=True)
wwtp_combined = add_load_columns(wwtp_combined)

first_columns = ['wwtp_site', 'time', 'source_file']
remaining_columns = [column for column in wwtp_combined.columns if column not in first_columns]
wwtp_combined = wwtp_combined[first_columns + remaining_columns]

print(wwtp_combined['wwtp_site'].value_counts().sort_index())


## 3. Overview of the combined WWTP data

In [ ]:
def count_missing_15min_steps(times):
    """Count missing 15-minute steps between the first and last timestamp."""
    valid_times = pd.Series(times).dropna().drop_duplicates().sort_values()

    if valid_times.empty:
        return None

    expected_times = pd.date_range(valid_times.iloc[0], valid_times.iloc[-1], freq='15min')
    return len(expected_times.difference(valid_times))


overview_rows = []

for site, group in wwtp_combined.groupby('wwtp_site'):
    overview_rows.append({
        'wwtp_site': site,
        'rows': len(group),
        'first_time': group['time'].min(),
        'last_time': group['time'].max(),
        'missing_15min_steps_between_first_and_last': count_missing_15min_steps(group['time']),
        'rows_with_reconstructed_time': int(group['time_reconstructed_from_concentration_sheet'].fillna(False).astype(bool).sum()),
        'flow_missing_rows': int(group['flow_m3d'].isna().sum()),
        'cod_missing_rows': int(group['cod_mgO2_L'].isna().sum()),
        'tn_missing_rows': int(group['tn_mgN_L'].isna().sum()),
        'tp_missing_rows': int(group['tp_mgP_L'].isna().sum()),
        'mean_flow_m3d': group['flow_m3d'].mean(),
        'total_volume_m3_2017': group['volume_m3_15min'].sum(),
        'total_cod_kg_2017': group['cod_kg_15min'].sum(),
        'total_tn_kg_2017': group['tn_kg_15min'].sum(),
        'total_tp_kg_2017': group['tp_kg_15min'].sum(),
        'max_cod_mgO2_L': group['cod_mgO2_L'].max(),
        'max_tn_mgN_L': group['tn_mgN_L'].max(),
        'max_tp_mgP_L': group['tp_mgP_L'].max(),
    })

site_summary = pd.DataFrame(overview_rows).sort_values('wwtp_site').reset_index(drop=True)
site_summary


## 4. Read WWTP discharge locations and candidate MPS coordinates

In [ ]:
wwtp_locations = pd.read_excel(LOZINGEN_FILE, sheet_name='RWZI')

wwtp_locations = wwtp_locations.rename(columns={
    'RWZI': 'rwzi_name',
    'Zuiv.gebied': 'wwtp_site',
    'x(m)': 'x_wwtp',
    'y(m)': 'y_wwtp',
    'Ontvangende waterloop': 'receiving_watercourse',
    'Deelbekken': 'subbasin',
    'Koppeling in ICM ? ': 'coupled_in_icm',
    'Opmerkingen': 'remarks',
})

wwtp_locations = wwtp_locations[wwtp_locations['wwtp_site'].isin(WWTP_FILES.keys())].copy()
wwtp_locations = wwtp_locations.sort_values('wwtp_site').reset_index(drop=True)
wwtp_locations


In [ ]:
mps_raw = gpd.read_file(MPS_GPKG_FILE)

mps_raw['latitude'] = pd.to_numeric(mps_raw['station_latitude'], errors='coerce')
mps_raw['longitude'] = pd.to_numeric(mps_raw['station_longitude'], errors='coerce')

mps_project = mps_raw[
    mps_raw['longitude'].between(5.20, 5.60)
    & mps_raw['latitude'].between(51.10, 51.35)
].copy()

mps_lambert = mps_project.to_crs(31370)
mps_project['x_mps'] = mps_lambert.geometry.x
mps_project['y_mps'] = mps_lambert.geometry.y

mps_project = mps_project[[
    'station_name', 'station_no', 'station_id',
    'longitude', 'latitude', 'x_mps', 'y_mps',
]].sort_values('station_name').reset_index(drop=True)

mps_project


## 5. Build provisional MP01-MP07 candidate coordinates

The exact 2017 MP01-MP07 measurement coordinates are still not confirmed. Therefore this remains a candidate mapping.


In [ ]:
mp_measurements = pd.DataFrame([
    {'mps_station_id': 'MP01', 'mps_location_name': 'Goudbergstraat', 'mps_watercourse': 'Dommel'},
    {'mps_station_id': 'MP02', 'mps_location_name': 'Hoksentstraat', 'mps_watercourse': 'Dommel'},
    {'mps_station_id': 'MP03', 'mps_location_name': 'Watermolen van Molhem', 'mps_watercourse': 'Dommel'},
    {'mps_station_id': 'MP04', 'mps_location_name': 'Warmbeek downstream of canal', 'mps_watercourse': 'Warmbeek'},
    {'mps_station_id': 'MP06', 'mps_location_name': 'Warmbeek upstream of Prinsenloop', 'mps_watercourse': 'Warmbeek'},
    {'mps_station_id': 'MP07', 'mps_location_name': 'Eindergatloop', 'mps_watercourse': 'Eindergatloop'},
])

candidate_rows = []

for _, mp in mp_measurements.iterrows():
    candidate_mps = mps_project[
        mps_project['station_name'].str.contains(mp['mps_watercourse'], case=False, na=False)
    ]

    for _, candidate in candidate_mps.iterrows():
        candidate_rows.append({
            'mps_station_id': mp['mps_station_id'],
            'mps_location_name': mp['mps_location_name'],
            'mps_watercourse': mp['mps_watercourse'],
            'mps_candidate_station_name': candidate['station_name'],
            'mps_candidate_station_no': candidate['station_no'],
            'x_mps': candidate['x_mps'],
            'y_mps': candidate['y_mps'],
            'mapping_status': 'candidate_only_mp_coordinate_not_confirmed',
        })

mp_coordinate_candidates = pd.DataFrame(candidate_rows)
mp_coordinate_candidates


## 6. Spatial mapping: WWTP discharge point to candidate MP location

This uses straight-line distance in Lambert 72. It is useful for screening, but not final routing.


In [ ]:
distance_rows = []

for _, wwtp in wwtp_locations.iterrows():
    for _, mp in mp_coordinate_candidates.iterrows():
        dx = wwtp['x_wwtp'] - mp['x_mps']
        dy = wwtp['y_wwtp'] - mp['y_mps']
        distance_m = (dx ** 2 + dy ** 2) ** 0.5

        distance_rows.append({
            'wwtp_site': wwtp['wwtp_site'],
            'rwzi_name': wwtp['rwzi_name'],
            'x_wwtp': wwtp['x_wwtp'],
            'y_wwtp': wwtp['y_wwtp'],
            'receiving_watercourse': wwtp['receiving_watercourse'],
            'mps_station_id': mp['mps_station_id'],
            'mps_location_name': mp['mps_location_name'],
            'mps_watercourse': mp['mps_watercourse'],
            'mps_candidate_station_name': mp['mps_candidate_station_name'],
            'mps_candidate_station_no': mp['mps_candidate_station_no'],
            'distance_m': round(distance_m, 1),
            'mapping_status': 'straight_line_candidate_only_not_final_routing',
        })

wwtp_to_mp_distances = pd.DataFrame(distance_rows)

wwtp_to_mp_nearest = (
    wwtp_to_mp_distances
    .sort_values('distance_m')
    .groupby(['wwtp_site', 'mps_station_id'], as_index=False)
    .first()
    .sort_values(['wwtp_site', 'mps_station_id'])
    .reset_index(drop=True)
)

nearest_mps_candidate = (
    wwtp_to_mp_distances
    .sort_values('distance_m')
    .groupby('wwtp_site', as_index=False)
    .first()
    .sort_values('wwtp_site')
    .reset_index(drop=True)
)

nearest_mps_candidate[['wwtp_site', 'rwzi_name', 'mps_candidate_station_name', 'distance_m']]


## 7. Save outputs

In [ ]:
wwtp_combined.to_csv(OUTPUT_COMBINED_CSV, index=False)
wwtp_to_mp_nearest.to_csv(OUTPUT_DISTANCE_CSV, index=False)

assumptions = pd.DataFrame([
    {'topic': 'Input files', 'remark': 'Only the five RWZI workbooks are used. Separate effluentASM/effluentTN/effluentTP files are not used because their contents are already present in the RWZI workbooks.'},
    {'topic': 'Sheets used', 'remark': "For most WWTPs the sheets are 'alle data', 'TP', 'TN', 'COD'. Lommel uses 'tot P', 'tot N' and 'tot CZV'."},
    {'topic': 'Time reconstruction', 'remark': "Eksel has invalid year-0000 timestamps in 'alle data'. These are reconstructed from the TP concentration sheet."},
    {'topic': 'Spatial mapping', 'remark': 'Distances to MP01-MP07 are provisional because exact 2017 MPS measurement coordinates are not yet confirmed.'},
    {'topic': 'Routing', 'remark': 'Straight-line distances are screening information only. Final routing still needs river topology, upstream/downstream relation and residence time.'},
])

with pd.ExcelWriter(OUTPUT_EXCEL, engine='xlsxwriter') as writer:
    site_summary.round(3).to_excel(writer, sheet_name='Site_Summary', index=False)
    wwtp_locations.to_excel(writer, sheet_name='WWTP_Locations', index=False)
    mps_project.to_excel(writer, sheet_name='MPS_GPKG_Candidates', index=False)
    mp_coordinate_candidates.to_excel(writer, sheet_name='MP_Coordinate_Candidates', index=False)
    nearest_mps_candidate.to_excel(writer, sheet_name='Nearest_MPS_Candidate', index=False)
    wwtp_to_mp_nearest.to_excel(writer, sheet_name='WWTP_to_MP_Candidates', index=False)
    wwtp_combined.head(1000).round(4).to_excel(writer, sheet_name='Combined_15min_Sample', index=False)
    assumptions.to_excel(writer, sheet_name='Assumptions', index=False)

print('Saved:', OUTPUT_EXCEL)
print('Saved:', OUTPUT_COMBINED_CSV)
print('Saved:', OUTPUT_DISTANCE_CSV)
